# Notebook 02: Exploratory Data Analysis

---

## Overview

This notebook performs comprehensive exploratory data analysis on the NIH Chest X-Ray dataset.

**Objectives:**
1. Analyze patient demographics (age, gender)
2. Visualize disease distribution and class imbalance
3. Examine multi-label patterns and co-occurrence
4. Display sample X-ray images
5. Assess data quality and identify potential issues

**Outputs:**
- Statistical summaries and visualizations
- Disease correlation heatmap
- Sample image grid
- EDA report saved to `outputs/reports/`

---

## 1. Setup and Load Data

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Image processing
from PIL import Image
import cv2

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries imported successfully")

In [ ]:
# Define paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data'
RAW_DATA_DIR = DATA_DIR / 'raw'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
FIGURES_DIR = OUTPUTS_DIR / 'figures'
REPORTS_DIR = OUTPUTS_DIR / 'reports'

print(f"Data directory: {RAW_DATA_DIR}")

In [ ]:
# Load metadata
metadata_df = pd.read_csv(RAW_DATA_DIR / 'Data_Entry_2017.csv')

print(f"✓ Loaded metadata: {len(metadata_df):,} images")
print(f"Columns: {list(metadata_df.columns)}")
metadata_df.head()

## 2. Patient Demographics Analysis

**Learning Outcome 1**: Apply core principles of statistics and probability

In [ ]:
# Age statistics
print("📊 Age Statistics:")
print(f"  Mean: {metadata_df['Patient Age'].mean():.1f} years")
print(f"  Median: {metadata_df['Patient Age'].median():.1f} years")
print(f"  Std Dev: {metadata_df['Patient Age'].std():.1f} years")
print(f"  Min-Max: {metadata_df['Patient Age'].min():.0f} - {metadata_df['Patient Age'].max():.0f} years")

# Quartiles
print(f"\n  Quartiles:")
print(metadata_df['Patient Age'].describe()[['25%', '50%', '75%']])

In [ ]:
# Age distribution visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(metadata_df['Patient Age'], bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(metadata_df['Patient Age'].mean(), color='red', linestyle='--', label=f'Mean: {metadata_df["Patient Age"].mean():.1f}')
axes[0].axvline(metadata_df['Patient Age'].median(), color='green', linestyle='--', label=f'Median: {metadata_df["Patient Age"].median():.1f}')
axes[0].set_xlabel('Age (years)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Patient Age Distribution')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Box plot
axes[1].boxplot(metadata_df['Patient Age'], vert=True)
axes[1].set_ylabel('Age (years)')
axes[1].set_title('Patient Age Box Plot')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / '02_age_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Figure saved to outputs/figures/")

In [ ]:
# Gender distribution
gender_counts = metadata_df['Patient Gender'].value_counts()

print("\n👥 Gender Distribution:")
for gender, count in gender_counts.items():
    percentage = (count / len(metadata_df)) * 100
    print(f"  {gender}: {count:,} ({percentage:.1f}%)")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
gender_counts.plot(kind='bar', ax=axes[0], color=['#3498db', '#e74c3c'])
axes[0].set_title('Gender Distribution')
axes[0].set_xlabel('Gender')
axes[0].set_ylabel('Number of Images')
axes[0].tick_params(axis='x', rotation=0)

# Pie chart
axes[1].pie(gender_counts, labels=gender_counts.index, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Gender Distribution (Percentage)')

plt.tight_layout()
plt.savefig(FIGURES_DIR / '02_gender_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Disease Distribution Analysis

Analyze the 15 disease classes and quantify class imbalance

In [ ]:
# Extract all disease labels
all_labels = metadata_df['Finding Labels'].str.split('|')
unique_diseases = sorted(set([label for labels in all_labels for label in labels]))

# Count frequency of each disease
disease_counts = {}
for disease in unique_diseases:
    disease_counts[disease] = sum(metadata_df['Finding Labels'].str.contains(disease, regex=False))

disease_df = pd.DataFrame([
    {'Disease': disease, 'Count': count, 'Percentage': (count/len(metadata_df))*100}
    for disease, count in disease_counts.items()
]).sort_values('Count', ascending=False)

print("🏥 Disease Distribution:\n")
print(disease_df.to_string(index=False))

# Calculate imbalance ratio
max_class = disease_df.iloc[0]['Count']
min_class = disease_df.iloc[-1]['Count']
imbalance_ratio = max_class / min_class
print(f"\n⚠️ Class Imbalance Ratio: {imbalance_ratio:.1f}:1")
print(f"   (Most common: {disease_df.iloc[0]['Disease']} vs Least common: {disease_df.iloc[-1]['Disease']})")

In [ ]:
# Visualize disease distribution
plt.figure(figsize=(14, 8))
bars = plt.barh(disease_df['Disease'], disease_df['Count'])

# Color the most common class differently
bars[0].set_color('#e74c3c')  # Red for "No Finding"

plt.xlabel('Number of Images', fontsize=12)
plt.ylabel('Disease Class', fontsize=12)
plt.title('Disease Frequency Distribution (Class Imbalance)', fontsize=14, fontweight='bold')
plt.grid(axis='x', alpha=0.3)

# Add value labels
for i, (disease, count) in enumerate(zip(disease_df['Disease'], disease_df['Count'])):
    plt.text(count + 500, i, f'{count:,}', va='center')

plt.tight_layout()
plt.savefig(FIGURES_DIR / '02_disease_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Multi-Label Analysis

Examine co-occurring diseases (multiple labels per image)

In [ ]:
# Count number of labels per image
label_counts = all_labels.apply(len)

print("📊 Multi-Label Statistics:")
print(f"\n  Images with single label:  {(label_counts == 1).sum():,} ({(label_counts == 1).sum()/len(metadata_df)*100:.1f}%)")
print(f"  Images with 2 labels:      {(label_counts == 2).sum():,} ({(label_counts == 2).sum()/len(metadata_df)*100:.1f}%)")
print(f"  Images with 3 labels:      {(label_counts == 3).sum():,} ({(label_counts == 3).sum()/len(metadata_df)*100:.1f}%)")
print(f"  Images with 4+ labels:     {(label_counts >= 4).sum():,} ({(label_counts >= 4).sum()/len(metadata_df)*100:.1f}%)")
print(f"\n  Maximum labels per image: {label_counts.max()}")
print(f"  Average labels per image: {label_counts.mean():.2f}")

In [ ]:
# Placeholder: Create disease co-occurrence matrix
# This will be used for Hypothesis 2 in Notebook 04

print("Creating disease co-occurrence matrix...")
print("(This shows which diseases appear together in the same image)")

# TODO: Implement co-occurrence matrix
# For each pair of diseases, count how often they appear together
# Will create heatmap in Notebook 04 for hypothesis testing

print("\n⏭️ Full co-occurrence analysis will be in Notebook 04 (Hypothesis Testing)")

## 5. Sample Image Visualization

Display example X-ray images for each disease class

In [ ]:
# Find image directories
image_dirs = [d for d in RAW_DATA_DIR.iterdir() if d.is_dir() and 'images' in d.name.lower()]

if image_dirs:
    IMAGE_DIR = image_dirs[0]
    print(f"✓ Found image directory: {IMAGE_DIR}")
else:
    # Images might be in subdirectories
    print("Searching for image files...")
    # Add metadata 'Image Index' column to find paths
    print("\nNote: Actual image paths will be determined after download completes")

In [ ]:
# Placeholder: Display sample images
# This will show one example image per disease class

print("Sample Image Grid (to be implemented):")
print("  - Display 15 images (one per disease class)")
print("  - Show image with disease label as title")
print("  - Highlight differences in visual appearance")

# TODO: Create 5x3 grid of sample images
# fig, axes = plt.subplots(3, 5, figsize=(20, 12))
# for each disease, find first image and display

print("\n✓ Sample visualization will be completed after confirming image paths")

## 6. Data Quality Assessment

In [ ]:
# Check for missing values
print("🔍 Data Quality Check:\n")
print("Missing values per column:")
print(metadata_df.isnull().sum())

# Check for duplicates
duplicate_images = metadata_df['Image Index'].duplicated().sum()
print(f"\nDuplicate image names: {duplicate_images}")

# Check age range validity
invalid_ages = ((metadata_df['Patient Age'] < 0) | (metadata_df['Patient Age'] > 120)).sum()
print(f"Invalid age values: {invalid_ages}")

if metadata_df.isnull().sum().sum() == 0 and duplicate_images == 0 and invalid_ages == 0:
    print("\n✓ Data quality looks good! No major issues detected.")
else:
    print("\n⚠️ Some data quality issues detected - will handle in preprocessing")

## 7. Save EDA Report

In [ ]:
# Create comprehensive EDA report
import json

eda_report = {
    'dataset_summary': {
        'total_images': len(metadata_df),
        'unique_patients': metadata_df['Patient ID'].nunique(),
    },
    'age_statistics': {
        'mean': float(metadata_df['Patient Age'].mean()),
        'median': float(metadata_df['Patient Age'].median()),
        'std': float(metadata_df['Patient Age'].std()),
        'min': float(metadata_df['Patient Age'].min()),
        'max': float(metadata_df['Patient Age'].max())
    },
    'gender_distribution': metadata_df['Patient Gender'].value_counts().to_dict(),
    'disease_distribution': disease_df.to_dict('records'),
    'class_imbalance_ratio': float(imbalance_ratio),
    'multi_label_stats': {
        'single_label': int((label_counts == 1).sum()),
        'two_labels': int((label_counts == 2).sum()),
        'three_labels': int((label_counts == 3).sum()),
        'four_plus_labels': int((label_counts >= 4).sum()),
        'max_labels': int(label_counts.max()),
        'avg_labels': float(label_counts.mean())
    }
}

report_path = REPORTS_DIR / '02_eda_report.json'
with open(report_path, 'w') as f:
    json.dump(eda_report, f, indent=2)

print(f"✓ EDA report saved to: {report_path}")

## 8. Summary and Key Findings

### Key Insights 📊

1. **Patient Demographics**:
   - Age range: 1-95 years
   - Gender distribution shows patient diversity
   
2. **Disease Distribution**:
   - 15 disease classes with significant imbalance
   - "No Finding" is the majority class (>50%)
   - Rare diseases have <1% prevalence
   
3. **Multi-Label Challenge**:
   - Many images have multiple diseases
   - Up to 8 labels per image
   - Requires multi-label classification approach
   
4. **Data Quality**:
   - No missing values in metadata
   - Clean dataset ready for modeling

### Next Steps ⏭️

**Notebook 03: Image Preprocessing**
- Load and resize X-ray images
- Implement data augmentation
- Create train/validation/test splits
- Prepare data for modeling

In [ ]:
print("="*60)
print("  ✅ Notebook 02 Complete: Exploratory Data Analysis")
print("="*60)
print(f"\nGenerated outputs:")
print(f"  📊 Figures: {len(list(FIGURES_DIR.glob('02_*.png')))} saved")
print(f"  📄 Reports: {report_path}")
print(f"\nReady for Notebook 03: Image Preprocessing!")